In [1]:
import random
import time

import pandas as pd
from generate_recipe_task import *
from experiment_classes import WrongIngTest

num_recipes = 5

actor_specs = ["Alice", "She", 1, 0.5, 0]
transcipt_steps = 3
max_recipes = 4

recipe_df = pd.read_csv("csv/bulk_formatted.csv", index_col=0)

num_steps = [len(row.iloc[5].split('\n')) for i, row in recipe_df.iterrows()]
num_ings = [len(row.iloc[3].split('\n')) for i, row in recipe_df.iterrows()]
recipe_df['num_steps'] = num_steps
recipe_df['num_ings'] = num_ings

long_recipes = recipe_df[recipe_df['num_steps'] >= 5]

long_recipes

,title,ingredients,instructions,formatted_ings,ing_list,transformations,num_steps,num_ings
0,Egg Mcmuffin,"[""1 large egg (grade A)"", ""1 English muffin"", ...","[""You will also need an Egg Ring, or a tuna ca...","egg = Ingredient('eg01', 'egg', 50, over='too ...","ingredients = [egg, muffin, bacon, cheese, but...","t1 = Transformation('fry', 'Toast the english ...",15,7
2,Hibiscus Sangria,"[""1 cup Grapes, Washed And De-stemmed"", ""1 cup...","[""1. Place the grapes on a parchment-lined pla...","grapes = Ingredient('grap', 'Grapes, Washed An...","ingredients = [grapes, water, hibiscus, sugar,...","t1 = Transformation('chill', 'Place the grapes...",5,8
3,Chocolate Oatmeal Cookies,"[""2 c. sugar"", ""1/2 c. milk"", ""1 stick butter ...","[""Combine all ingredients, except oatmeal, in ...","sugar = Ingredient('sgr1', 'sugar', 400, over=...","ingredients = [sugar, milk, butter, cocoa, pea...","t1 = Transformation('mix', 'Combine all ingred...",5,7
5,Pumpkin And Vegetable Risotto,"[""1 onion, chopped finely"", ""1 red capsicum, d...","[""Put pumpkin on to steam before continuing (i...","onion = Ingredient('onio', 'onion, chopped fin...","ingredients = [onion, capsicum, corn, carrot, ...","t1 = Transformation('boil', 'Put pumpkin on to...",9,18
8,Ham And Dill Party Dip,"[""1 1/3 c. mayonnaise"", ""1 1/3 c. sour cream"",...","[""Mix all ingredients, except bread."", ""Chill ...","mayonnaise = Ingredient('mayo', 'mayonnaise', ...","ingredients = [mayonnaise, sourcream, driedoni...","t1 = Transformation('mix', 'Mix all ingredient...",5,7
...,...,...,...,...,...,...,...,...
194,Pumpkin Cake Roll,"[""1/2 c. flour"", ""1 tsp. baking powder"", ""2 ts...","[""Combine flour, baking powder and pumpkin pie...","flour = Ingredient('flou', 'flour', 60, over='...","ingredients = [flour, bakingpowder, pumpkinspi...","t1 = Transformation('mix', 'Combine flour, bak...",17,11
195,Snack Taco Dip,"[""2 (8 oz.) cream cheese"", ""1 medium salsa (16...","[""Blend with mixer the cream cheese and salsa....","creamcheese = Ingredient('crmc', 'cream cheese...","ingredients = [creamcheese, salsa, lettuce, to...","t1 = Transformation('mix', 'Blend with mixer t...",5,8
196,No Fat Oven-Fried Squash,"[""4 or 5 squash"", ""corn meal"", ""fat-free butte...","[""Wash and prepare squash. Coat both sides of ...","squash = Ingredient('sqsh', 'squash', 900, ove...","ingredients = [squash, cornmeal, spray]","t1 = Transformation('mix', 'Coat both sides of...",5,4
197,Crunchy Vegetable Stuffed Zucchini,"[""2 zucchini (medium 6-8-inch)"", ""1 tablespoon...","[""Preheat oven to 350."", ""Line an 9 or 9\"" squ...","zucchini = Ingredient('zuch', 'zucchini (mediu...","ingredients = [zucchini, oliveoil, mushroom, o...","t1 = Transformation('bake', 'Preheat oven to 3...",14,11


In [23]:
# samples = random.sample(range(len(long_recipes)), k=num_recipes)
unused_col = []
accuracies_col = []
aug_ings = ['10 oz garam masala', '5 cloves star anise']
old_output = ''
batch = []
# for recipe_id in range(len(long_recipes)):
for recipe_id in [10, ]:
    selected_row = long_recipes.iloc[recipe_id]
    print(f'Generating transcript for {selected_row.iloc[0]}')
    generate_task_file_from_row(selected_row, actor_specs, transcipt_steps)
    # time.sleep(1)
    !python 'recipe_task/recipe_task.py'
    # time.sleep(1)
    output_text = open("recipe_task/output.txt", "r").read()

    if output_text == old_output:
        print('generation error')
        unused_col.append([])
        accuracies_col.append([])
        print('='*50)
        print()
        continue
    old_output = output_text

    output_text = output_text.replace('_', ' ')
    print(output_text)

    try:
        test = WrongIngTest(output_text, transcipt_steps, selected_row, actor_specs, max_recipes=max_recipes, augmented_ingredients=aug_ings)
    except:
        print('test creation error')
        unused_col.append([])
        accuracies_col.append([])
        print('='*50)
        continue
    unused_col.append(test.unused_ingredients)
    if len(test.unused_ingredients) < test.num_options - 1:
        print('not enough ingredients')
        accuracies_col.append([])
        print('='*50)
        print()
        continue
    batch.extend(test.prepare_batch(n=1))
    print(test.perturb_recipe())
    # test.run_test(n=num_recipes)
    # test.print_results()
    # accuracies = test.get_results()
    # print(accuracies)
    # accuracies_col.append(accuracies)
    # print(test.reasoning)

    print('='*50)
    print()

print(len(batch))
batch[0]

Generating transcript for Mini Caramel Apples
Alice boils the granulated sugar to make some caramel.
Alice stirs the caramel and butter to make some caramel sauce.
Then, she mixes the apples and caramel sauce to make some caramel apples.

Recipe 1
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '3 tbsp sugar pearl sprinkles', '10 oz garam masala', '5 cloves star anise']

Recipe 2
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '10 oz garam masala', '5 cloves star anise']

Recipe 3
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '3 tbsp sugar pearl sprinkles', '5 cloves star anise']

Recipe 4
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '3 tbsp sugar pearl sprinkles', '10 oz garam masala']



3


'{"custom_id": "replace", "method": "POST", "url": "/v1/responses""body": {"model": "gpt-5", "reasoning": [{"effort": medium}], instructions=Below are a list of recipes Alice may be attempting to make, followed by their ingredients: \n Recipe 1\n [\'2 None apples, cut into 16 balls using a melon baller\', \'1 1/2 cups granulated sugar\', \'1 tbsp butter\', \'3 tbsp sugar pearl sprinkles\', \'10 oz garam masala\', \'5 cloves star anise\']\n\nRecipe 2\n [\'2 None apples, cut into 16 balls using a melon baller\', \'1 1/2 cups granulated sugar\', \'1 tbsp butter\', \'10 oz garam masala\', \'5 cloves star anise\']\n\nRecipe 3\n [\'2 None apples, cut into 16 balls using a melon baller\', \'1 1/2 cups granulated sugar\', \'1 tbsp butter\', \'3 tbsp sugar pearl sprinkles\', \'5 cloves star anise\']\n\nRecipe 4\n [\'2 None apples, cut into 16 balls using a melon baller\', \'1 1/2 cups granulated sugar\', \'1 tbsp butter\', \'3 tbsp sugar pearl sprinkles\', \'10 oz garam masala\']\n\nGiven a tra

In [ ]:
# with open("batchinput.jsonl", "w") as f:
#     for i, query in enumerate(batch):
#         query = query.replace('replace', f'query_{i}')
#         query = query.replace("\n", "\\n")
#         query = query.replace("\'", "\\'")
#         f.write(query)


In [ ]:
results = long_recipes[['title', 'ingredients', 'directions', 'num_steps', 'num_ings']]

results.insert(len(results.columns), 'unused_ingredients',  unused_col)
results.insert(len(results.columns), 'prediction_accuracy',  accuracies_col)
results = results.loc[[len(x) > 0 for x in results['prediction_accuracy']]]
results

In [ ]:
results.to_csv(f'csv/full_max_recipes_{max_recipes}.csv')

In [ ]:
# dfs = [pd.read_csv(f'csv/bulk_formatted_{i+1}.csv', names=['title', 'ingredients', 'instructions', 'formatted_ings', 'ing_list', 'transformations']) for i in range(4)]
# combined = pd.concat(dfs, ignore_index=True)
# combined.nunique()

In [ ]:
# combined.to_csv(f'csv/bulk_formatted.csv')

In [39]:
from openai import OpenAI
client = OpenAI()

rt = """
Recipe 2
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '10 oz garam masala', '5 cloves star anise']

Recipe 3
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '3 tbsp sugar pearl sprinkles', '5 cloves star anise']

Recipe 4
 ['2 None apples, cut into 16 balls using a melon baller', '1 1/2 cups granulated sugar', '1 tbsp butter', '3 tbsp sugar pearl sprinkles', '10 oz garam masala']
 """

for i in range(num_recipes):
    response = client.responses.create(
        model="gpt-5",
        instructions=f"Below are a list of recipes Alice may be attempting to make, followed by their ingredients: \n"
                                                     f"{rt}"
                                                     f"Given a transcript of actions performed by Alice, output the name of the recipe Alice is trying to cook.",
        input="""Alice stirs the caramel and butter to make some caramel sauce.\nThen, she mixes the apples and caramel sauce to make some caramel apples.\nAlice appears shocked after she mixes in the sugar pearl sprinkles.""",
        reasoning={
            "effort": "low",
            "summary": "auto"
        }
    )

    print(response.output_text)
    for summ in response.output[0].summary:
        print(summ.text)

Recipe 3
**Analyzing recipe choices**

I’m trying to choose a recipe based on actions from a transcript. There are three recipes with ingredient lists. To make caramel apples, ingredients include granulated sugar, butter, and apples. Then, sugar pearl sprinkles are mixed in, which surprises someone. The recipes include sugar pearl sprinkles in both recipes 3 and 4. Recipe 3 has star anise, while recipe 4 uses garam masala. However, there's no evidence of either spice being used, just the sprinkles causing the shock.
**Rethinking recipe consistency**

Since garam masala wasn't used, it implies Recipe 4 could be incorrect. The transcript doesn’t mention any spice addition at all. The shock could suggest an accidental addition of sugar pearl sprinkles instead of garam masala? But considering the text says she’s shocked after mixing in the sprinkles, maybe it’s because garnishing with sprinkles clashes with garam masala's savory flavor. Alternatively, perhaps star anise would have been mor